# 01 - Exploration et Compréhension des Données

## Objectifs de ce notebook
- Charger et comprendre le dataset Titanic
- Explorer la structure des données
- Identifier les valeurs manquantes
- Comprendre chaque variable

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Chargement des données

Le dataset provient de [Kaggle Titanic Competition](https://www.kaggle.com/competitions/titanic).

Pour télécharger les données :
1. Aller sur la page Kaggle
2. Télécharger `train.csv` et `test.csv`
3. Les placer dans `data/raw/`

In [ ]:
# Charger les données
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

print(f"Taille du dataset d'entraînement : {train_df.shape}")
print(f"Taille du dataset de test : {test_df.shape}")

## 2. Aperçu des données

In [ ]:
# Premières lignes
train_df.head(10)

In [ ]:
# Dernières lignes
train_df.tail()

In [ ]:
# Échantillon aléatoire
train_df.sample(5, random_state=RANDOM_STATE)

## 3. Description des variables

| Variable | Définition | Type | Valeurs |
|----------|------------|------|----------|
| PassengerId | Identifiant unique | int | - |
| Survived | **Variable cible** (survie) | int | 0 = Non, 1 = Oui |
| Pclass | Classe du billet | int | 1 = 1ère, 2 = 2ème, 3 = 3ème |
| Name | Nom du passager | str | - |
| Sex | Sexe | str | male, female |
| Age | Âge en années | float | - |
| SibSp | Nombre de frères/sœurs/conjoints à bord | int | - |
| Parch | Nombre de parents/enfants à bord | int | - |
| Ticket | Numéro du billet | str | - |
| Fare | Prix du billet | float | - |
| Cabin | Numéro de cabine | str | - |
| Embarked | Port d'embarquement | str | C = Cherbourg, Q = Queenstown, S = Southampton |

In [ ]:
# Types de données
print("Types de données :")
print(train_df.dtypes)

In [ ]:
# Informations générales
train_df.info()

## 4. Statistiques descriptives

In [ ]:
# Variables numériques
train_df.describe()

In [ ]:
# Variables catégorielles
train_df.describe(include=['object'])

## 5. Analyse des valeurs manquantes

In [ ]:
# Compter les valeurs manquantes
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
}).sort_values('Valeurs manquantes', ascending=False)

print("Valeurs manquantes par colonne :")
missing_df[missing_df['Valeurs manquantes'] > 0]

In [ ]:
# Visualisation des valeurs manquantes
fig, ax = plt.subplots(figsize=(10, 6))

missing_cols = missing_df[missing_df['Valeurs manquantes'] > 0]
colors = ['#ff6b6b' if pct > 50 else '#ffd93d' if pct > 10 else '#6bcb77' 
          for pct in missing_cols['Pourcentage (%)']]

bars = ax.barh(missing_cols.index, missing_cols['Pourcentage (%)'], color=colors)
ax.set_xlabel('Pourcentage de valeurs manquantes (%)')
ax.set_title('Valeurs manquantes dans le dataset')

# Ajouter les pourcentages sur les barres
for bar, pct in zip(bars, missing_cols['Pourcentage (%)']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'{pct:.1f}%', va='center')

plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analyse de la variable cible (Survived)

In [ ]:
# Distribution de la variable cible
survived_counts = train_df['Survived'].value_counts()
survived_pct = train_df['Survived'].value_counts(normalize=True) * 100

print("Distribution de Survived :")
print(f"  - Décédés (0) : {survived_counts[0]} ({survived_pct[0]:.1f}%)")
print(f"  - Survivants (1) : {survived_counts[1]} ({survived_pct[1]:.1f}%)")

In [ ]:
# Visualisation
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#ff6b6b', '#6bcb77']
labels = ['Décédé', 'Survivant']

bars = ax.bar(labels, survived_counts.values, color=colors, edgecolor='black')
ax.set_ylabel('Nombre de passagers')
ax.set_title('Distribution de la survie sur le Titanic')

# Ajouter les valeurs sur les barres
for bar, count, pct in zip(bars, survived_counts.values, survived_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
            f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/survival_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Valeurs uniques par colonne

In [ ]:
# Nombre de valeurs uniques
for col in train_df.columns:
    n_unique = train_df[col].nunique()
    print(f"{col}: {n_unique} valeurs uniques")
    
    # Afficher les valeurs si peu nombreuses
    if n_unique <= 10:
        print(f"   → {train_df[col].unique()}")

## 8. Résumé et insights

### Ce qu'on a appris :

1. **Taille du dataset** : 891 passagers avec 12 colonnes

2. **Variable cible** : `Survived` (binaire) - environ 38% de survivants

3. **Valeurs manquantes** :
   - `Cabin` : ~77% manquant → difficile à utiliser directement
   - `Age` : ~20% manquant → nécessite imputation
   - `Embarked` : ~0.2% manquant → facile à gérer

4. **Types de variables** :
   - Numériques : Age, SibSp, Parch, Fare
   - Catégorielles : Sex, Embarked, Pclass
   - Texte (à transformer) : Name, Ticket, Cabin
   - ID : PassengerId

### Prochaines étapes :
- EDA approfondie (notebook 02)
- Stratégie pour les valeurs manquantes
- Feature engineering potentiel